In [1]:
import pandas as pd 

In [2]:
df=pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.isnull().sum()
df.shape

(50000, 2)

In [4]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

In [5]:

#Preprocessing 
df["review"]=df["review"].str.lower()

# Preprocessing 

In [6]:
import re 

1. Remove URL

In [7]:
def remove_url(text):
    text=re.sub(r"https\S+","",text)
    return text
df["review"]=df["review"].apply(remove_url)

2. Remove Punctuations

In [8]:
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]=df["review"].apply(remove_punctuations)

3. Removing HTML

In [9]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

4. Removing Stopwords

In [10]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sroy9\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sroy9\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sroy9\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [12]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text
df["review"]=df["review"].apply(remove_stopwords)


In [13]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


5. Stemming 

In [14]:
from nltk.stem import PorterStemmer

In [15]:
def stemming(text):
    ps=PorterStemmer()
    tokens=word_tokenize(text)
    stemmed_words=[]

    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)

6. Vectorization 

In [16]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


7. Encoding

In [17]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [18]:
y=df["sentiment"]

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])


In [20]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057136 stored elements and shape (49582, 5000)>

# Dataset and Dataloaders

In [21]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [22]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [23]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [24]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [25]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

# Building RNN

In [26]:
import torch.nn as nn
import torch.optim as optim

In [28]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=126,num_layer=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layer=num_layer
        #RNN Layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layer,batch_first=True)
        #FC layer
        self.fc=nn.Linear(hidden_size,1) #1 O/P for Many to One architecture
    def forward(self,x):
        h0=torch.zeros(self.num_layer,x.size(0),self.hidden_size)
        #initializes the 1st step with zero 

        out,_=self.rnn(x,h0)
        #1st value-> hidden state of all timesteps
        out=self.fc(out[:,-1,:]) #fc layer converts the 128 values from the out to a single O/P
        return out  

In [29]:
input_size=X_train.shape[1]
model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

# Training the RNN

In [36]:
epochs=10
for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb=Xb.unsqueeze(1) # Adds an additional layer to make it 3d
        outputs=model(Xb) # O/P-> (batch_size,1)
        outputs=torch.sigmoid(outputs.squeeze()) 
        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()
    print(f"Epoch= {epoch+1}/{epochs} and loss={loss.item()}") 

Epoch= 1/10 and loss=0.28108644485473633
Epoch= 2/10 and loss=0.19235415756702423
Epoch= 3/10 and loss=0.2551265060901642
Epoch= 4/10 and loss=0.38622626662254333
Epoch= 5/10 and loss=0.27946269512176514
Epoch= 6/10 and loss=0.22675146162509918
Epoch= 7/10 and loss=0.21003296971321106
Epoch= 8/10 and loss=0.13175247609615326
Epoch= 9/10 and loss=0.09728755056858063
Epoch= 10/10 and loss=0.3332071006298065


Evaluation 

In [42]:
model.eval()

with torch.no_grad():
    correct_values=0
    tot_vals=0

for Xb,yb in test_loader:
    Xb=Xb.unsqueeze(1)
    outputs=model(Xb)
    predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()
    tot_vals+=yb.size(0)
    correct_values+=(predicted==yb).sum().item()
print(f"Accuracy={(correct_values/tot_vals)*100}%")

Accuracy=85.21730362004638%
